In [0]:
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/bronze/raw/"
FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

for table, fname in FILES.items():
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)   # keep everything as string in bronze
          .option("multiLine", True)
          .option("escape", '"')
          .csv(BASE + fname))
    # some CSVs carry a hidden BOM character in the first column name
    df = df.toDF(*[c.replace("\ufeff", "").strip() for c in df.columns])
    df = (df.withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_source_file", F.lit(fname)))
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"workspace.bronze.{table}"))
    print(table, df.count())

In [0]:
from pyspark.sql import functions as F
B, S = "workspace.bronze", "workspace.silver"

def save(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
      .saveAsTable(f"{S}.{name}")
    print(name, df.count())

# ---- orders: cast timestamps, remove duplicate order_ids
o = spark.table(f"{B}.orders")
for c in ["order_purchase_timestamp", "order_approved_at",
          "order_delivered_carrier_date", "order_delivered_customer_date",
          "order_estimated_delivery_date"]:
    o = o.withColumn(c, F.to_timestamp(c))
save(o.dropDuplicates(["order_id"]), "orders")

# ---- order_items: cast numerics and dates
i = (spark.table(f"{B}.order_items")
     .withColumn("order_item_id", F.col("order_item_id").cast("int"))
     .withColumn("price", F.col("price").cast("double"))
     .withColumn("freight_value", F.col("freight_value").cast("double"))
     .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date"))
     .dropDuplicates(["order_id", "order_item_id"]))
save(i, "order_items")

# ---- customers and sellers: trim text, standardize state
c = (spark.table(f"{B}.customers")
     .withColumn("customer_city", F.initcap(F.trim("customer_city")))
     .withColumn("customer_state", F.upper(F.trim("customer_state")))
     .dropDuplicates(["customer_id"]))
save(c, "customers")

s = (spark.table(f"{B}.sellers")
     .withColumn("seller_city", F.initcap(F.trim("seller_city")))
     .withColumn("seller_state", F.upper(F.trim("seller_state")))
     .dropDuplicates(["seller_id"]))
save(s, "sellers")

# ---- products: fix typos in column names, cast, translate category
p = (spark.table(f"{B}.products")
     .withColumnRenamed("product_name_lenght", "product_name_length")
     .withColumnRenamed("product_description_lenght", "product_description_length"))
for col in ["product_name_length", "product_description_length", "product_photos_qty",
            "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    p = p.withColumn(col, F.col(col).cast("int"))
t = spark.table(f"{B}.category_translation").drop("_ingested_at", "_source_file")
p = (p.join(t, "product_category_name", "left")
      .withColumn("category", F.coalesce("product_category_name_english",
                                         "product_category_name", F.lit("unknown")))
      .dropDuplicates(["product_id"]))
save(p, "products")